<a href="https://colab.research.google.com/github/aaronseymour7/MLQ/blob/main/full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install git+https://github.com/qiskit-community/mps-to-circuit.git
!pip install "jax[cuda12_pip]" \
            optax \
            flax \
!pip install numpy matplotlib
!pip install physics-tenpy
!pip install qiskit

  Cloning https://github.com/qiskit-community/mps-to-circuit.git to /tmp/pip-req-build-xjnlfux1
  Running command git clone --filter=blob:none --quiet https://github.com/qiskit-community/mps-to-circuit.git /tmp/pip-req-build-xjnlfux1
  Resolved https://github.com/qiskit-community/mps-to-circuit.git to commit 0b0b26e565f3341abb7c1036cd80df0f2680f4d1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 940.0/940.0 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.3/258.3 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [3]:
# @title
from __future__ import annotations

import numpy as np
from typing import Protocol, runtime_checkable

try:
    import tenpy.networks.site as tsite
    from tenpy.models.lattice import Chain, Square, Triangular, Honeycomb, Kagome
except ImportError:
    raise ImportError("pip install physics-tenpy")


@runtime_checkable
class BaseLattice(Protocol):
    name:      str
    n_sites:   int
    nn_edges:  list[tuple[int, int]]
    nnn_edges: list[tuple[int, int]]


class TenpyLattice:
    def __init__(self, lat, name: str, pbc: bool = True,
                 nnn_key: str | None = "next_nearest_neighbors"):
        self._lat    = lat          # keep the full TeNPy object for downstream use
        self.name    = name
        self.pbc     = pbc
        self.n_sites = lat.N_sites  # use TeNPy's own count, not order.shape[0]

        dim   = lat.dim
        order = lat.order           # (N, dim+1): [x, y, ..., u]

        # Build index map with plain Python ints to avoid numpy scalar hash issues
        self._idx: dict[tuple, int] = {
            tuple(int(v) for v in row): i for i, row in enumerate(order)
        }

        self.nn_edges  = self._build_edges("nearest_neighbors")
        self.nnn_edges = self._build_edges(nnn_key) if nnn_key else []

        self._validate()

    def _build_edges(self, pair_key: str) -> list[tuple[int, int]]:
        if pair_key not in self._lat.pairs:
            return []

        bonds = self._lat.pairs[pair_key]
        dim   = self._lat.dim
        Ls    = np.asarray(self._lat.Ls, dtype=int)

        seen:  set[tuple[int, int]] = set()
        edges: list[tuple[int, int]] = []

        for site_row in self._lat.order:
            xy1 = site_row[:dim].astype(int)   # spatial coords
            u1  = int(site_row[dim])            # unit-cell index

            for bond in bonds:
                bu1 = int(bond[0])
                bu2 = int(bond[1])
                dxy = np.asarray(bond[2], dtype=int).reshape(dim)

                if bu1 != u1:
                    continue

                xy2_raw = xy1 + dxy

                if self.pbc:
                    xy2 = xy2_raw % Ls
                else:
                    if np.any(xy2_raw < 0) or np.any(xy2_raw >= Ls):
                        continue
                    xy2 = xy2_raw

                key2 = tuple(xy2.tolist()) + (bu2,)
                j    = self._idx.get(key2, -1)
                if j < 0:
                    continue

                i    = self._idx[tuple(int(v) for v in site_row)]
                edge = (min(i, j), max(i, j))
                if edge[0] != edge[1] and edge not in seen:
                    seen.add(edge)
                    edges.append(edge)

        edges.sort()
        return edges

    def _validate(self):
        """Cross-check edge count against known coordination numbers."""
        N   = self.n_sites
        nn  = len(self.nn_edges)
        nnn = len(self.nnn_edges)

        expected_z = {
            'chain':       (2,  2),
            'square':      (4,  4),
            'triangular':  (6,  6),
            'honeycomb':   (3,  6),
            'kagome':      (4,  4),
        }
        kind = self.name.split()[0]
        if kind not in expected_z or not self.pbc:
            return

        # Skip validation for lattices so small that PBC causes bond collapse
        # (e.g. 2×2 square: the +x and -x neighbours of a site are the same site).
        # This is a degenerate geometry, not a bug in edge-building.
        Ls = self._lat.Ls                         # e.g. (2, 2) for a 2×2 square
        if any(l < 3 for l in Ls):
            print(f"[{self.name}]  skipping validation: lattice too small "
                  f"(Ls={list(Ls)}) for PBC edge-count check.")
            return

        z_nn, z_nnn  = expected_z[kind]
        expected_nn  = N * z_nn  // 2
        expected_nnn = N * z_nnn // 2

        if nn != expected_nn:
            raise ValueError(
                f"{self.name}: expected {expected_nn} nn edges (z={z_nn}), "
                f"got {nn}. Lattice geometry is wrong."
            )
        if nnn != expected_nnn:
            raise ValueError(
                f"{self.name}: expected {expected_nnn} nnn edges (z={z_nnn}), "
                f"got {nnn}. Lattice geometry is wrong."
            )

        print(f"[{self.name}]  nn={nn} ({z_nn}/site)  "
              f"nnn={nnn} ({z_nnn}/site)  ✓")

    @property
    def tenpy_lat(self):
        """The raw TeNPy lattice object, for Schmidt spectra etc."""
        return self._lat

    def __repr__(self):
        return (f"TenpyLattice(name={self.name!r}, N={self.n_sites}, "
                f"nn={len(self.nn_edges)}, nnn={len(self.nnn_edges)})")


def _site():
    return tsite.SpinHalfSite(conserve="Sz")


def _bc(pbc: bool, dim: int) -> list[str] | str:
    """TeNPy bc: string for 1D, list for 2D."""
    if dim == 1:
        return "periodic" if pbc else "open"
    return ["periodic"] * dim if pbc else ["open"] * dim


def make_lattice(kind: str, L: int = 8, pbc: bool = True) -> TenpyLattice:
    kind = kind.lower()
    site = _site()

    if kind == "chain":
        lat  = Chain(L, site, bc=_bc(pbc, 1))
        name = f"chain L={L}"
        nnn  = "next_nearest_neighbors"

    elif kind == "square":
        Lx, Ly = _factor2(L, n_uc=1, prefer_square=True)
        lat    = Square(Lx, Ly, site, bc=_bc(pbc, 2))
        name   = f"square {Lx}x{Ly}"
        nnn    = "next_nearest_neighbors"

    elif kind == "triangular":
        Lx, Ly = _factor2(L, n_uc=1, prefer_square=True)
        lat    = Triangular(Lx, Ly, site, bc=_bc(pbc, 2))
        name   = f"triangular {Lx}x{Ly}"
        nnn    = "next_nearest_neighbors"

    elif kind == "honeycomb":
        Lx, Ly = _factor2(L, n_uc=2, prefer_square=True)
        lat    = Honeycomb(Lx, Ly, site, bc=_bc(pbc, 2))
        name   = f"honeycomb {Lx}x{Ly}"
        nnn    = "next_nearest_neighbors"

    elif kind == "kagome":
        Lx, Ly = _factor2(L, n_uc=3, prefer_square=True)
        lat    = Kagome(Lx, Ly, site, bc=_bc(pbc, 2))
        name   = f"kagome {Lx}x{Ly}"
        nnn    = "next_nearest_neighbors"

    else:
        raise ValueError(
            f"Unknown lattice {kind!r}. "
            "Choose: chain, square, triangular, honeycomb, kagome"
        )

    return TenpyLattice(lat, name=name, pbc=pbc, nnn_key=nnn)


def _factor2(n_total: int, n_uc: int, prefer_square: bool = True) -> tuple[int, int]:
    if n_total % n_uc != 0:
        raise ValueError(f"{n_total} not divisible by unit-cell size {n_uc}")
    sites = n_total // n_uc
    best  = None
    for Ly in range(1, int(sites**0.5) + 1):
        if sites % Ly == 0:
            best = (sites // Ly, Ly)
    if best is None:
        raise ValueError(f"Cannot factorise {sites}")
    return best

In [11]:
# @title
"""
UCJ (Unitary Cluster Jastrow) — optimised circuit builder.

Entry point:  build_ucj(n, lattice, j1, j2, ...)
Returns:      (transpiled QuantumCircuit, gate_counts, depth)

Gate conventions:
  _qk_xy        : imaginary Givens  = RXX(-θ) · RYY(-θ)
  _qk_xy_phased : real Givens       = RZ(-π/2) · RXX(-θ) · RYY(-θ) · RZ(+π/2)
  CPhaseGate(φ) : Jastrow           = diag(1,1,1,e^{iφ})
"""

from __future__ import annotations
import warnings
import functools
import numpy as np
from scipy.optimize import minimize as scipy_minimize
from scipy.sparse.linalg import eigsh, LinearOperator

import jax
import jax.numpy as jnp

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import RXXGate, RYYGate, RZGate, CPhaseGate
#from lattices import BaseLattice, make_lattice

# =============================================================================
# GLOBAL CONFIG
# =============================================================================
J1              = 1.0
J2              = 0.0
VARIANT         = "re"        # 're' | 'im' | 'g'
K_LAYERS        = 1
N_RESTARTS      = 1
NOISE_SCALE     = 0.05
SEED            = 23

LBFGS_MAXITER   = 800
LBFGS_MAXFUN    = 50_000
LBFGS_FTOL      = 1e-14
LBFGS_GTOL      = 1e-8

TARGET_BASIS    = ["cx", "rz", "h", "s", "sdg"]
OPT_LEVEL       = 3           # Qiskit transpiler optimisation level (0-3)

# =============================================================================
# JAX SETUP
# =============================================================================
jax.config.update("jax_enable_x64", True)
_DEVICE = jax.devices("cpu")[0]

def _to_device(x):
    return jax.device_put(x, _DEVICE)


# =============================================================================
# EXACT DIAGONALISATION
# =============================================================================
def _get_n_up(n: int) -> int:
    return n // 2


def _build_basis(n: int, n_up: int) -> np.ndarray:
    return np.array([b for b in range(1 << n) if bin(b).count("1") == n_up],
                    dtype=np.int64)


def _build_hamiltonian_op(n, n_up, nn_edges, nnn_edges, j1, j2):
    basis   = _build_basis(n, n_up)
    idx_map = {int(b): i for i, b in enumerate(basis)}
    dim     = len(basis)
    rows, cols, vals = [], [], []

    for edges, j in [(nn_edges, j1), (nnn_edges, j2)]:
        for si, sj in edges:
            for row, bits in enumerate(basis):
                zi = 0.5 if (bits >> si) & 1 else -0.5
                zj = 0.5 if (bits >> sj) & 1 else -0.5
                rows.append(row); cols.append(row)
                vals.append(j * zi * zj)
                if ((bits >> si) & 1) != ((bits >> sj) & 1):
                    fl  = bits ^ (1 << si) ^ (1 << sj)
                    col = idx_map.get(int(fl), -1)
                    if col >= 0:
                        rows.append(row); cols.append(col)
                        vals.append(0.5 * j)

    rows_np = np.array(rows, dtype=np.int32)
    cols_np = np.array(cols, dtype=np.int32)
    vals_np = np.array(vals, dtype=np.float64)

    def matvec(v):
        out = np.zeros(dim, dtype=v.dtype)
        np.add.at(out, rows_np, vals_np * v[cols_np])
        return out

    op = LinearOperator((dim, dim), matvec=matvec, dtype=np.float64)
    return op, basis, idx_map


def get_ground_state(n, n_up, nn_edges, nnn_edges, j1, j2):
    op, basis, idx_map = _build_hamiltonian_op(
        n, n_up, nn_edges, nnn_edges, j1, j2)
    evals, evecs = eigsh(op, k=1, which="SA", tol=1e-10, maxiter=10_000)
    e_exact      = float(evals[0])
    psi_exact    = evecs[:, 0] / np.linalg.norm(evecs[:, 0])
    return e_exact, psi_exact, basis, idx_map


# =============================================================================
# JAX HAMILTONIAN
# =============================================================================
def build_jax_hamiltonian(n, n_up, nn_edges, nnn_edges, j1, j2, basis, idx_map):
    rows, cols, vals = [], [], []
    for edges, j in [(nn_edges, j1), (nnn_edges, j2)]:
        for si, sj in edges:
            for row, bits in enumerate(basis):
                zi = 0.5 if (bits >> si) & 1 else -0.5
                zj = 0.5 if (bits >> sj) & 1 else -0.5
                rows.append(row); cols.append(row); vals.append(j * zi * zj)
                if ((bits >> si) & 1) != ((bits >> sj) & 1):
                    fl = bits ^ (1 << si) ^ (1 << sj)
                    if fl in idx_map:
                        rows.append(row)
                        cols.append(idx_map[fl])
                        vals.append(0.5 * j)

    h_rows = _to_device(jnp.array(rows, dtype=jnp.int32))
    h_cols = _to_device(jnp.array(cols, dtype=jnp.int32))
    h_vals = _to_device(jnp.array(vals, dtype=jnp.float64))

    dim = len(basis)

    @functools.partial(jax.jit)
    def apply_H(psi):
        return (jnp.zeros(dim, dtype=psi.dtype)
                .at[h_rows].add(h_vals * psi[h_cols]))

    return apply_H


# =============================================================================
# JASTROW
# =============================================================================
def build_jastrow_fn(n, basis):
    pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
    pi    = _to_device(jnp.array([p[0] for p in pairs], dtype=jnp.int32))
    pj    = _to_device(jnp.array([p[1] for p in pairs], dtype=jnp.int32))
    bits  = _to_device(jnp.array(basis, dtype=jnp.int32))

    @jax.jit
    def jastrow_phase(theta_J):
        def acc(phase, args):
            tk, i, j = args
            bi = ((bits >> i) & 1).astype(jnp.float64)
            bj = ((bits >> j) & 1).astype(jnp.float64)
            return phase + tk * bi * bj, None
        phase, _ = jax.lax.scan(
            acc,
            jnp.zeros(bits.shape[0], dtype=jnp.float64),
            (theta_J, pi, pj),
        )
        return phase

    return jastrow_phase


def apply_jastrow(psi, theta_J, jastrow_phase_fn):
    return psi * jnp.exp(1j * jastrow_phase_fn(theta_J))


# =============================================================================
# GIVENS PAIRS
# =============================================================================
def build_givens_pairs(n, basis, idx_map):
    srcs_ragged, dsts_ragged = [], []
    for i in range(n):
        for j in range(i + 1, n):
            srcs, dsts = [], []
            for row, bits in enumerate(basis):
                if ((bits >> i) & 1) and not ((bits >> j) & 1):
                    fl = bits ^ (1 << i) ^ (1 << j)
                    if fl in idx_map:
                        srcs.append(row)
                        dsts.append(idx_map[fl])
            srcs_ragged.append(np.array(srcs, dtype=np.int32))
            dsts_ragged.append(np.array(dsts, dtype=np.int32))

    counts  = np.array([len(s) for s in srcs_ragged], dtype=np.int32)
    row_ptr = np.zeros(len(counts) + 1, dtype=np.int32)
    row_ptr[1:] = np.cumsum(counts)

    srcs_cat = np.concatenate(srcs_ragged) if srcs_ragged else np.array([], dtype=np.int32)
    dsts_cat = np.concatenate(dsts_ragged) if dsts_ragged else np.array([], dtype=np.int32)

    return (_to_device(jnp.array(srcs_cat, dtype=jnp.int32)),
            _to_device(jnp.array(dsts_cat, dtype=jnp.int32)),
            row_ptr)


# =============================================================================
# ANSATZ STATE
# =============================================================================
def _givens_scan(psi, thetas, srcs, dsts, row_ptr, imag=False):
    for k in range(row_ptr.shape[0] - 1):
        s, e = int(row_ptr[k]), int(row_ptr[k + 1])
        if s == e:
            continue
        c, ss = jnp.cos(thetas[k]), jnp.sin(thetas[k])
        ps, pd = psi[srcs[s:e]], psi[dsts[s:e]]
        if imag:
            ns, nd = c * ps - 1j * ss * pd, -1j * ss * ps + c * pd
        else:
            ns, nd = c * ps - ss * pd, ss * ps + c * pd
        psi = psi.at[srcs[s:e]].set(ns).at[dsts[s:e]].set(nd)
    return psi


def _ucj_state(theta, variant, k_layers, psi0, n_pair, srcs, dsts, row_ptr, jastrow_fn):
    psi    = psi0
    stride = 3 * n_pair if variant == "g" else 2 * n_pair
    for l in range(k_layers):
        off = l * stride
        psi = apply_jastrow(psi, theta[off:off + n_pair], jastrow_fn)
        psi = _givens_scan(psi, theta[off + n_pair:off + 2 * n_pair],
                           srcs, dsts, row_ptr, imag=(variant == "im"))
        if variant == "g":
            psi = _givens_scan(psi, theta[off + 2 * n_pair:off + 3 * n_pair],
                               srcs, dsts, row_ptr, imag=True)
    return psi


def _energy(psi, apply_H):
    norm = jnp.dot(jnp.conj(psi), psi)
    return jnp.real(jnp.dot(jnp.conj(psi), apply_H(psi)) / norm)


# =============================================================================
# OPTIMISATION
# =============================================================================
def _make_val_grad(variant, k_layers, psi0, n_pair, srcs, dsts, row_ptr,
                   jastrow_fn, apply_H):
    def efn(theta):
        psi = _ucj_state(theta, variant, k_layers, psi0, n_pair,
                         srcs, dsts, row_ptr, jastrow_fn)
        return _energy(psi, apply_H)

    return jax.jit(jax.value_and_grad(efn))


def _optimise(val_grad_fn, x0):
    x0_gpu = _to_device(jnp.array(x0, dtype=jnp.float64))
    val_grad_fn(x0_gpu)  # warm-up JIT

    def scipy_fn(x_np):
        x_gpu = _to_device(jnp.array(x_np, dtype=jnp.float64))
        E, g  = val_grad_fn(x_gpu)
        return float(E), np.array(g, dtype=np.float64)

    result = scipy_minimize(
        scipy_fn, x0, jac=True, method="L-BFGS-B",
        options={"maxiter": LBFGS_MAXITER, "maxfun": LBFGS_MAXFUN,
                 "ftol": LBFGS_FTOL, "gtol": LBFGS_GTOL})
    return np.array(result.x), float(result.fun), result


def _run_optimisation(variant, k_layers, n, n_pair, psi_neel,
                      srcs, dsts, row_ptr, jastrow_fn, apply_H, e_exact):
    val_grad_fn = _make_val_grad(variant, k_layers, psi_neel, n_pair,
                                 srcs, dsts, row_ptr, jastrow_fn, apply_H)
    stride = 3 * n_pair if variant == "g" else 2 * n_pair

    best_params, best_E = None, np.inf
    rng = np.random.default_rng(SEED)

    for restart in range(N_RESTARTS):
        x0 = NOISE_SCALE * rng.standard_normal(k_layers * stride)
        opt_x, opt_E, result = _optimise(val_grad_fn, x0)
        print(f"  [restart {restart}]  E={opt_E:.8f}  "
              f"|ΔE|={abs(opt_E - e_exact):.4e}  "
              f"nit={result.nit}  nfev={result.nfev}")
        if opt_E < best_E:
            best_E, best_params = opt_E, opt_x

    return best_params, best_E


# =============================================================================
# QISKIT GATE HELPERS
# =============================================================================
def _qk_xy(qc, theta, q0, q1):
    qc.append(RXXGate(-theta), [q0, q1])
    qc.append(RYYGate(-theta), [q0, q1])


def _qk_xy_phased(qc, theta, q0, q1):
    qc.append(RZGate(-np.pi / 2), [q1])
    qc.append(RXXGate(-theta), [q0, q1])
    qc.append(RYYGate(-theta), [q0, q1])
    qc.append(RZGate(np.pi / 2), [q1])


# =============================================================================
# CIRCUIT BUILDER
# =============================================================================
def _build_circuit(n, k_layers, variant, params, pairs):
    n_pair = len(pairs)
    stride = 3 * n_pair if variant == "g" else 2 * n_pair

    qc = QuantumCircuit(n)
    for i in range(0, n, 2):
        qc.x(i)

    for l in range(k_layers):
        off  = l * stride
        tJ   = params[off           : off + n_pair]
        tK_r = params[off + n_pair  : off + 2 * n_pair]
        tK_i = params[off + 2*n_pair: off + 3 * n_pair] if variant == "g" else None

        for k, (i, j) in enumerate(pairs):
            qc.append(CPhaseGate(float(tJ[k])), [i, j])

        if variant == "im":
            for k, (i, j) in enumerate(pairs):
                _qk_xy(qc, float(tK_r[k]), j, i)
        else:
            for k, (i, j) in enumerate(pairs):
                _qk_xy_phased(qc, float(tK_r[k]), j, i)
            if variant == "g":
                for k, (i, j) in enumerate(pairs):
                    _qk_xy(qc, float(tK_i[k]), j, i)

    return qc


# =============================================================================
# ENTRY POINT
# =============================================================================
def build_ucj(
    lattice,
    j1: float = J1,
    j2: float = J2,
    *,
    variant: str       = VARIANT,
    k_layers: int      = K_LAYERS,
    pairs: list | None = None,
    basis_gates: list | None = None,
) -> tuple[QuantumCircuit, dict[str, int], int]:
    """
    Optimise UCJ parameters then build + transpile the circuit.

    Parameters
    ----------
    lattice     : object with .nn_edges and .nnn_edges
    j1, j2      : Heisenberg couplings
    variant     : 're' | 'im' | 'g'
    k_layers    : number of UCJ layers
    pairs       : (i,j) qubit pairs; defaults to all upper-triangle pairs
    basis_gates : transpile target; defaults to TARGET_BASIS

    Returns
    -------
    tqc         : transpiled QuantumCircuit with optimised parameters
    gate_counts : dict[str, int]
    depth       : int
    """
    n = lattice.n_sites

    if pairs is None:
        pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
    if basis_gates is None:
        basis_gates = TARGET_BASIS

    n_pair = len(pairs)
    n_up   = _get_n_up(n)

    # ── exact diagonalisation ─────────────────────────────────────────────────
    print(f"\n[build_ucj]  N={n}  variant={variant}  k={k_layers}  "
          f"J1={j1}  J2={j2}")
    print("  Running exact diagonalisation...")
    e_exact, _, basis, idx_map = get_ground_state(
        n, n_up, lattice.nn_edges, lattice.nnn_edges, j1, j2)
    print(f"  E_exact = {e_exact:.10f}  (E/site = {e_exact/n:.10f})")

    # ── JAX structures ────────────────────────────────────────────────────────
    apply_H     = build_jax_hamiltonian(
        n, n_up, lattice.nn_edges, lattice.nnn_edges, j1, j2, basis, idx_map)
    jastrow_fn  = build_jastrow_fn(n, basis)
    srcs, dsts, row_ptr = build_givens_pairs(n, basis, idx_map)

    # Néel initial state
    neel_bits = sum(1 << i for i in range(n) if i % 2 == 0)
    psi_neel  = _to_device(
        jnp.zeros(len(basis), dtype=jnp.complex128)
        .at[idx_map[neel_bits]].set(1.0)
    )

    # ── optimisation ─────────────────────────────────────────────────────────
    print(f"  Optimising ({N_RESTARTS} restart(s))...")
    best_params, best_E = _run_optimisation(
        variant, k_layers, n, n_pair, psi_neel,
        srcs, dsts, row_ptr, jastrow_fn, apply_H, e_exact)
    print(f"  Best E = {best_E:.10f}  |ΔE| = {abs(best_E - e_exact):.4e}")

    # ── build + transpile circuit ─────────────────────────────────────────────
    qc = _build_circuit(n, k_layers, variant, best_params, pairs)
    try:
        tqc = transpile(qc, basis_gates=basis_gates,
                        optimization_level=OPT_LEVEL)
    except Exception as exc:
        warnings.warn(f"transpile failed ({exc}); returning undecomposed circuit")
        tqc = qc

    gate_counts = dict(tqc.count_ops())
    depth       = tqc.depth()

    print(f"\n  [Circuit]  depth={depth}  total_gates={sum(gate_counts.values())}")
    for gate, count in sorted(gate_counts.items(), key=lambda x: -x[1]):
        print(f"    {gate:<20} {count}")

    return tqc, gate_counts, depth

'''
# =============================================================================
# DEMO
# =============================================================================
if __name__ == "__main__":
    lattice = make_lattice('kagome', L=n)

    tqc, counts, depth = build_ucj(lattice, j1=1.0, j2=0.0,
                                   variant="re", k_layers=1)
'''

'\n# =============================================================================\n# DEMO\n# =============================================================================\nif __name__ == "__main__":\n    lattice = make_lattice(\'kagome\', L=n)\n\n    tqc, counts, depth = build_ucj(lattice, j1=1.0, j2=0.0,\n                                   variant="re", k_layers=1)\n'

In [16]:
# @title
"""
dmrg_mps_circuit.py
====================
DMRG ground state via TeNPy → Qiskit circuit + Hamiltonian + basis.

The TeNPy model is now driven entirely by a BaseLattice object so that
arbitrary geometries (chain, square, triangular, honeycomb, kagome, …)
are handled correctly.  The coupling topology comes from
  lattice.nn_edges   → J1 terms
  lattice.nnn_edges  → J2 terms
and the TeNPy lattice object (lattice.tenpy_lat) is used directly so
that the MPS bond structure respects the canonical MPS ordering that
TeNPy chose when it built the lattice.

Returns
-------
  run() → (exact_qc, approx_qc, H_sparse, basis)

Dependencies
------------
    pip install physics-tenpy qiskit scipy numpy
    pip install git+https://github.com/qiskit-community/mps-to-circuit.git
"""

from __future__ import annotations

import numpy as np
from scipy.sparse import lil_matrix, csr_matrix
from qiskit import transpile

# ── TeNPy ────────────────────────────────────────────────────────────────────
try:
    import tenpy
    from tenpy.networks.site import SpinHalfSite
    from tenpy.models.model import CouplingMPOModel
    from tenpy.networks.mps import MPS
    from tenpy.algorithms import dmrg
    print(f"[TeNPy]   {tenpy.__version__}")
except ImportError:
    raise ImportError("pip install physics-tenpy")

try:
    from mps_to_circuit import mps_to_circuit
except ImportError:
    raise ImportError(
        "pip install git+https://github.com/qiskit-community/mps-to-circuit.git"
    )

try:
    from qiskit import QuantumCircuit
    import qiskit
    print(f"[Qiskit]  {qiskit.__version__}")
except ImportError:
    raise ImportError("pip install qiskit")

#from lattices import BaseLattice

# =============================================================================
# CONFIG
# =============================================================================
J1           = 1.0
J2           = 0.0
DMRG_CHI_MAX = 256
DMRG_MIXER   = True
BASIS_GATES  = ["cx", "rz", "h", "s", "sdg"]

# =============================================================================
# HAMILTONIAN  (sparse, fixed-Sz sector)
# Used for energy checks and basis construction; topology from BaseLattice.
# =============================================================================
def build_basis(n: int) -> np.ndarray:
    n_up = n // 2
    return np.array([b for b in range(1 << n) if bin(b).count('1') == n_up],
                    dtype=np.int64)


def build_hamiltonian(
    lattice: BaseLattice,
    j1: float = J1,
    j2: float = J2,
) -> tuple[csr_matrix, np.ndarray]:
    """
    Return (H_sparse, basis) in the n//2-up sector.

    Edge topology is taken from lattice.nn_edges (J1) and
    lattice.nnn_edges (J2); no assumption about 1D chain structure.
    """
    n       = lattice.n_sites
    basis   = build_basis(n)
    idx_map = {int(b): i for i, b in enumerate(basis)}
    H       = lil_matrix((len(basis), len(basis)), dtype=np.float64)

    edge_sets = [(lattice.nn_edges, j1)]
    if j2 and lattice.nnn_edges:
        edge_sets.append((lattice.nnn_edges, j2))

    for edges, j in edge_sets:
        for si, sj in edges:
            for row, bits in enumerate(basis):
                zi = 0.5 if (bits >> si) & 1 else -0.5
                zj = 0.5 if (bits >> sj) & 1 else -0.5
                H[row, row] += j * zi * zj
                if ((bits >> si) & 1) != ((bits >> sj) & 1):
                    fl  = bits ^ (1 << si) ^ (1 << sj)
                    col = idx_map.get(int(fl), -1)
                    if col >= 0:
                        H[row, col] += 0.5 * j

    return csr_matrix(H), basis


# =============================================================================
# TENPY MODEL  —  topology-aware
# =============================================================================
class HeisenbergLattice(CouplingMPOModel):
    """
    Heisenberg J1-J2 model on an arbitrary geometry.

    model_params must contain:
        tenpy_lat  : the raw TeNPy lattice object  (from BaseLattice.tenpy_lat)
        nn_edges   : list[tuple[int,int]]  — nearest-neighbour pairs
        nnn_edges  : list[tuple[int,int]]  — next-nearest-neighbour pairs
        J1, J2     : float coupling constants
        conserve   : 'Sz' (default)
        bc_MPS     : 'finite' (default)

    Couplings are added site-by-site from the explicit edge lists, so
    any lattice geometry is handled correctly without relying on dx offsets.
    """

    def init_sites(self, model_params):
        return SpinHalfSite(conserve=model_params.get('conserve', 'Sz'))

    def init_lattice(self, model_params):
        # Reuse the pre-built TeNPy lattice so MPS ordering is consistent
        # with the one embedded in BaseLattice.
        return model_params['tenpy_lat']

    def init_terms(self, model_params):
        j1        = model_params.get('J1', 1.0)
        j2        = model_params.get('J2', 0.0)
        nn_edges  = model_params['nn_edges']
        nnn_edges = model_params.get('nnn_edges', [])

        for i, j in nn_edges:
            self._add_heisenberg(i, j, j1)

        if j2:
            for i, j in nnn_edges:
                self._add_heisenberg(i, j, j2)

    # ------------------------------------------------------------------
    def _add_heisenberg(self, i: int, j: int, coupling: float) -> None:
        """
        Add S_i · S_j = Sz_i Sz_j + ½(S+_i S-_j + S-_i S+_j).

        add_coupling_term(strength, i, op_i, j, op_j) accepts flat MPS
        site indices directly — no lattice-coordinate conversion needed.
        This is the correct API when i, j come from our flat edge lists.
        add_local_term / add_coupling both go through lat2mps_idx which
        expects (x, y, ..., u) tuples and raises IndexError on plain ints.
        """
        # add_coupling_term requires i < j
        if i > j:
            i, j = j, i
        self.add_coupling_term(coupling,       i, j, 'Sz', 'Sz')
        self.add_coupling_term(coupling / 2.0, i, j, 'Sp', 'Sm')
        self.add_coupling_term(coupling / 2.0, i, j, 'Sm', 'Sp')


# =============================================================================
# DMRG
# =============================================================================
def run_dmrg(
    lattice: BaseLattice,
    j1: float = J1,
    j2: float = J2,
    chi_max: int = DMRG_CHI_MAX,
) -> tuple[float, MPS]:
    """
    Run DMRG on the lattice topology defined by *lattice*.

    Parameters
    ----------
    lattice : BaseLattice
        Provides n_sites, nn_edges, nnn_edges, and the raw TeNPy lattice
        object (.tenpy_lat) so that both the Hamiltonian and the MPS bond
        structure reflect the true geometry.
    j1, j2  : Heisenberg couplings
    chi_max : maximum MPS bond dimension

    Returns
    -------
    E     : DMRG ground-state energy
    psi   : converged MPS
    """
    n          = lattice.n_sites
    tenpy_lat  = lattice.tenpy_lat   # raw TeNPy object with MPS ordering

    model_params = dict(
        tenpy_lat  = tenpy_lat,
        nn_edges   = lattice.nn_edges,
        nnn_edges  = lattice.nnn_edges,
        J1         = j1,
        J2         = j2,
        conserve   = 'Sz',
        bc_MPS     = 'finite',
    )
    model = HeisenbergLattice(model_params)

    # Néel initial state along the MPS chain
    initial_state = ['up' if i % 2 == 0 else 'down' for i in range(n)]
    psi = MPS.from_product_state(
        model.lat.mps_sites(), initial_state, bc='finite')

    eng = dmrg.TwoSiteDMRGEngine(psi, model, {
        'trunc_params': {'chi_max': chi_max, 'svd_min': 1e-10},
        'N_sweeps_check': 1,
        'mixer': DMRG_MIXER,
        'mixer_params': {'amplitude': 1e-4, 'decay': 1.2, 'disable_after': 50},
    })
    E, psi = eng.run()
    print(f"[DMRG]  lattice={lattice.name}  E={E:.10f}  "
          f"E/site={E/n:.10f}  chi_max={max(psi.chi)}")
    return E, psi


# =============================================================================
# MPS → CIRCUITS
# =============================================================================
def mps_to_circuits(
    psi_mps: MPS,
    n: int,
    n_approx_layers: int = 10,
) -> tuple[QuantumCircuit, QuantumCircuit]:
    """
    Return (exact_qc, approx_qc) transpiled to BASIS_GATES.
    mps_to_circuit expects tensors in 'lpr' shape (left-bond, physical, right-bond).
    """
    mps_arrays = [psi_mps.get_B(i).to_ndarray() for i in range(n)]

    exact_qc  = mps_to_circuit(mps_arrays, method="exact",       shape="lpr")
    approx_qc = mps_to_circuit(mps_arrays, method="approximate", shape="lpr",
                                num_layers=n_approx_layers)

    exact_qc  = _transpile(exact_qc,  "exact")
    approx_qc = _transpile(approx_qc, "approximate")
    return exact_qc, approx_qc


def _transpile(qc: QuantumCircuit, label: str) -> QuantumCircuit:
    qc_t = transpile(qc, basis_gates=BASIS_GATES, optimization_level=0)
    print(f"\n[circuit:{label}]  qubits={qc_t.num_qubits}  "
          f"depth={qc_t.depth()}  gates={qc_t.size()}")
    for gate, count in qc_t.count_ops().items():
        print(f"  {gate:<5} {count}")
    return qc_t


# =============================================================================
# ENTRY POINT
# =============================================================================
def run(
    lattice: BaseLattice,
    j1: float = J1,
    j2: float = J2,
    chi_max: int = DMRG_CHI_MAX,
    n_approx_layers: int = 10,
) -> tuple[QuantumCircuit, QuantumCircuit, csr_matrix, np.ndarray]:
    """
    Run DMRG on *lattice* and return (exact_qc, approx_qc, H_sparse, basis).

    Parameters
    ----------
    lattice         : BaseLattice  — geometry, edges, and TeNPy object
    j1, j2          : Heisenberg couplings
    chi_max         : DMRG bond dimension cap
    n_approx_layers : brick-wall layers for the approximate MPS circuit

    Returns
    -------
    exact_qc   : Qiskit circuit (exact MPS isometry)
    approx_qc  : Qiskit circuit (approximate, n_approx_layers brick-wall layers)
    H          : sparse Hamiltonian in the n//2-up sector
    basis      : integer basis states for that sector
    """
    n = lattice.n_sites
    print(f"\n[run]  lattice={lattice.name}  N={n}  J1={j1}  J2={j2}  chi_max={chi_max}")

    H, basis = build_hamiltonian(lattice, j1, j2)
    print(f"[H]  sector dim={len(basis)}")

    _, psi_mps = run_dmrg(lattice, j1, j2, chi_max)
    exact_qc, approx_qc = mps_to_circuits(psi_mps, n, n_approx_layers)

    return exact_qc, approx_qc, H, basis

'''
if __name__ == '__main__':
    from lattices import make_lattice
    lattice = make_lattice('chain', L=8)
    exact_qc, approx_qc, H, basis = run(lattice, j1=J1, j2=J2, chi_max=DMRG_CHI_MAX)
'''

[TeNPy]   1.1.0
[Qiskit]  2.4.1


"\nif __name__ == '__main__':\n    from lattices import make_lattice\n    lattice = make_lattice('chain', L=8)\n    exact_qc, approx_qc, H, basis = run(lattice, j1=J1, j2=J2, chi_max=DMRG_CHI_MAX)\n"

In [32]:
# @title
import numpy as np
from scipy import optimize as opt
import matplotlib.pyplot as plt
from typing import List, Dict, Optional, Tuple
from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import LieTrotter          # or SuzukiTrotter
from qiskit.quantum_info import SparsePauliOp

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import LieTrotter

# ----------------------------------------------------------------------
# Helper functions (unchanged)
# ----------------------------------------------------------------------
def fixtimes(times, totaltime):
    xf = totaltime / np.sum(np.abs(times))
    times[:] = xf * times
    return times

def unpack(timesphases):
    ndouble = len(timesphases)
    n = ndouble // 2
    return timesphases[:n].copy(), timesphases[n:].copy()

def timesconstraints(timesphases, total_time):
    times, _ = unpack(timesphases)
    return abs(np.sum(times) - total_time)

def probability_constraints(timesphases, energies, overlap):
    _, phases = unpack(timesphases)
    return np.prod(np.cos(phases)**2) - 0.9

def probability_constraintsb(timesphases, energies, overlap, pos):
    times, phases = unpack(timesphases)
    return np.prod((np.cos(energies[pos] * times + phases))**2) - 0.9

def new_func_v3(timesphases, energies, overlap):
    ndouble = len(timesphases)
    n = ndouble // 2
    times = timesphases[:n]
    phases = timesphases[n:]
    num = np.prod(np.cos(phases))
    cos_vals = np.cos(energies[:, None] * times + phases)
    den = np.prod(cos_vals**2, axis=1).sum()
    return abs(1 - num / np.sqrt(den))

def new_func_v4(timesphases, energies, overlap):
    ndouble = len(timesphases)
    n = ndouble // 2
    times = timesphases[:n]
    phases = timesphases[n:]
    num = 1.0
    cos_vals = np.cos(energies[:, None] * times)
    den = np.prod(cos_vals**2, axis=1).sum()
    return abs(1 - num / np.sqrt(den))


def new_func_v5(coeffs_sq, energies):
    """
    coeffs_sq : |<psi_k|psi_UCJ>|^2, shape (dim,)
    energies  : E_k, shape (dim,)
    Returns a scalar objective to minimize (0 = perfect filter).
    """
    def objective(timesphases):
        n = len(timesphases) // 2
        times  = timesphases[:n]
        phases = timesphases[n:]

        # cos(E_k * t_i + phi_i), shape (dim, n)
        cos_vals = np.cos(energies[:, None] * times + phases)

        # filter weight for each eigenstate: prod_i cos(...)^2
        weights = np.prod(cos_vals**2, axis=1)   # shape (dim,)

        # filtered overlap with GS (k=0) vs total
        gs_weight  = coeffs_sq[0] * weights[0]
        total      = np.dot(coeffs_sq, weights)

        return 1.0 - gs_weight / (total + 1e-30)

    return objective
# ----------------------------------------------------------------------
# FilterBuilder with evaluation & plotting
# ----------------------------------------------------------------------
class FilterBuilder:
    _METHODS = {
        "v3":  (new_func_v3,  probability_constraints,  False),
        "v3b": (new_func_v3,  probability_constraintsb, True),
        "v4":  (new_func_v4,  None,                     False),
        "v5":  (new_func_v5,  None,                     False),  # sentinel

    }

    def __init__(self, total_time, energies, overlap, a=4, b=15,
                 optimizer="SLSQP", maxiter=5000, ftol=1e-12, coeffs_sq=None):

        self.total_time = float(total_time)
        self.energies = np.asarray(energies, dtype=float)
        self.overlap = float(overlap)
        self.a = int(a)
        self.b = int(b)
        self.optimizer = optimizer
        self.opts = {"maxiter": maxiter, "ftol": ftol}
        self.coeffs_sq = coeffs_sq
    # ------------------------------------------------------------------
    def build(self, method="v4") -> List[Dict]:
        if method not in self._METHODS:
            raise ValueError(f"method must be one of {list(self._METHODS)}")

        if method == "v5":
            if self.coeffs_sq is None:
                raise ValueError("v5 requires coeffs_sq passed to FilterBuilder()")
            objective = new_func_v5(self.coeffs_sq, self.energies)
            extra_constr, needs_pos = None, False
        else:
            objective, extra_constr, needs_pos = self._METHODS[method]

        results = []

        for ntimes in range(self.a, self.b + 1):
            # initial guess
            times = np.ones(ntimes)
            for i in range(1, ntimes):
                times[i] = times[i - 1] / 2.0
            times = fixtimes(times, self.total_time)

            timesphases = np.zeros(2 * ntimes)
            timesphases[:ntimes] = times

            # bounds
            bnd_times  = [(0.0, self.total_time / 3.0)] * ntimes
            bnd_phases = [(-np.pi / 2, np.pi / 2)]      * ntimes
            bnds       = bnd_times + bnd_phases

            # constraints
            constraints = [
                {"type": "eq", "fun": timesconstraints, "args": (self.total_time,)}
            ]
            if extra_constr is not None:
                if needs_pos:
                    constraints.append(
                        {"type": "ineq", "fun": extra_constr,
                        "args": (self.energies, self.overlap, 0)}
                    )
                else:
                    constraints.append(
                        {"type": "ineq", "fun": extra_constr,
                        "args": (self.energies, self.overlap)}
                    )

            # ── optimize ──────────────────────────────────────────────────────────
            if method == "v5":
                # objective is already a closure; pass NO args
                res = opt.minimize(
                    objective,
                    x0=timesphases,
                    method=self.optimizer,
                    bounds=bnds,
                    constraints=constraints,
                    options=self.opts,
                    tol=1e-13,
                )
            else:
                res = opt.minimize(
                    objective,
                    x0=timesphases,
                    args=(self.energies, self.overlap),
                    method=self.optimizer,
                    bounds=bnds,
                    constraints=constraints,
                    options=self.opts,
                    tol=1e-13,
                )

            times_opt  = res.x[:ntimes]
            phases_opt = res.x[ntimes:]

            results.append({
                "ntimes":  ntimes,
                "times":   times_opt.copy(),
                "phases":  phases_opt.copy(),
                "fun":     float(res.fun),
                "success": bool(res.success),
                "message": str(res.message),
                "result":  res,
            })

            print(f"ntimes={ntimes:2d}  time={times_opt.sum():.6f}  "
                  f"fun={res.fun:.3e}  success={res.success}")

        return results

    # ------------------------------------------------------------------
    @staticmethod
    def apply_filter(times: np.ndarray, phases: np.ndarray, energies: np.ndarray, state: np.ndarray) -> Tuple[float, np.ndarray]:
        """
        Apply pulse sequence to a state (implements filterprint logic).
        Returns (fdiff, filtered_state)
        """
        f0 = state.copy()
        arg = np.zeros(len(energies))
        xscale = np.zeros(len(energies))

        for i in range(len(times)):
            arg[:] = times[i] * energies[:]
            xscale[:] = np.cos(phases[i]) * np.cos(arg[:]) - np.sin(phases[i]) * np.sin(arg[:])
            f0[:] = f0[:] * xscale[:]

        fnorm = 1.0 / np.sqrt(np.sum(f0**2))
        f0[:] = f0[:] * fnorm

        return f0, fnorm

    # ------------------------------------------------------------------
    def evaluate(
        self,
        results: List[Dict],
        gs_state: np.ndarray,
        trial_state: np.ndarray,
        plot: bool = True,
        highlight_pos: Optional[int] = None,
        ax: Optional[plt.Axes] = None,
    ) -> List[Dict]:
        """
        Evaluate all optimized filters on the trial state.

        Parameters
        ----------
        results : list of dicts from .build()
        gs_state : ground state vector (length = N_en)
        trial_state : initial trial state
        plot : whether to plot filtered states
        highlight_pos : index in energies to draw a red line
        ax : matplotlib axis (optional)

        Returns
        -------
        eval_results : list of dicts with fdiff, f0, etc.
        """
        if ax is None and plot:
            fig, ax = plt.subplots(figsize=(10, 6))

        eval_results = []
        for res in results:
            times = res["times"]
            phases = res["phases"]
            f0, fnorm = self.apply_filter(times, phases, self.energies, trial_state.copy())
            fidelity = float(np.dot(gs_state, f0))**2   # should approach 1
            fdiff = 1.0 - fidelity

            eval_results.append({
                "ntimes": res["ntimes"],
                "fdiff": fdiff,
                "f0": f0.copy(),
                "norm": fnorm,
                "times": times.copy(),
                "phases": phases.copy(),
            })

            print(f"ntimes={res['ntimes']:2d}  totaltime={times.sum():.6f}  fdiff={fdiff:.6e}")

            if plot:
                ax.plot(self.energies, f0, label=f"{res['ntimes']} pulses")

        if plot:
            if highlight_pos is not None:
                ax.axvline(x=self.energies[highlight_pos], color='r', linestyle='-', alpha=0.7, label=f'E[{highlight_pos}]')
            ax.set_ylim(-1, 1)
            ax.set_xlabel("Energy")
            ax.set_ylabel("Filtered State Amplitude")
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

        return eval_results

    # ------------------------------------------------------------------
    def build_and_evaluate(
        self,
        method="v4",
        gs_state=None,
        trial_state=None,
        highlight_pos: int = 100,
        plot: bool = True
    ):
        """
        One-liner: build → evaluate → (optionally) plot.
        """
        results = self.build(method)

        if gs_state is None or trial_state is None:
            N = len(self.energies)
            gs_state = np.zeros(N)
            gs_state[0] = 1.0

            trial_state = np.zeros(N)
            trial_state[0] = self.overlap
            trial_state[1:] = np.sqrt((1 - self.overlap**2) / (N - 1))

            print(f"check norm trial state: {np.linalg.norm(trial_state):.6f}")
            print(f"check overlap with gs: {np.dot(gs_state, trial_state):.6f} (target: {self.overlap})")

        return self.evaluate(results, gs_state, trial_state, plot=plot, highlight_pos=highlight_pos)








FIDELITY_THRESHOLD = 1e-6  # fun < this means fidelity > ~0.9999

def _best_filter(results, sector_sv, psi_exact, evecs, evals):
    """Pick filter with highest postsel_prob among near-perfect fidelity results."""
    good = [r for r in results if r["fun"] < FIDELITY_THRESHOLD]
    if not good:
        good = results  # fallback: take best available

    def postsel_prob(r):
        _, prob = post_filter_fidelity_analytic(
            sector_sv, psi_exact, evecs, evals,
            r["times"], r["phases"],
        )
        return prob

    return max(good, key=postsel_prob)




def build_filter_circuit(
    state_circuit: QuantumCircuit,   # your UCJ or DMRG circuit
    hamiltonian: SparsePauliOp,
    times: np.ndarray,
    phases: np.ndarray,
    trotter_steps: int = 1,
) -> QuantumCircuit:
    """
    Append the Hadamard-test filter to an existing state circuit.

    Returns a circuit with 1 ancilla qubit prepended.
    Ancilla qubit index = 0; system qubits = 1..n.

    Postselect all ancilla measurements on 0 to get the filtered state.
    """
    n = state_circuit.num_qubits

    anc = QuantumRegister(1, 'anc')
    sys = QuantumRegister(n, 'sys')
    cr  = ClassicalRegister(len(times), 'postsel')

    qc = QuantumCircuit(anc, sys, cr)

    # Prepare system in trial state
    qc.compose(state_circuit, qubits=list(range(1, n + 1)), inplace=True)

    for i, (t, phi) in enumerate(zip(times, phases)):
        # Step 1: ancilla to |+>
        qc.h(0)

        # Step 2: controlled-e^{-iHt} — ancilla controls system evolution
        evo = PauliEvolutionGate(
            hamiltonian,
            time=float(t),
            synthesis=LieTrotter(reps=trotter_steps),
        )
        # Make it controlled on ancilla qubit 0
        c_evo = evo.control(1)
        qc.append(c_evo, [0] + list(range(1, n + 1)))

        # Step 3: Rz(2*phi) on ancilla — this is the phase kick
        qc.rz(2 * float(phi), 0)

        # Step 4: H on ancilla
        qc.h(0)

        # Step 5: measure ancilla, postselect on 0
        qc.measure(0, i)

        # Reset ancilla for next pulse (only matters in hardware;
        # for simulation you branch on measurement outcome)
        qc.reset(0)

        qc.barrier()

    return qc




def post_filter_fidelity_analytic(
    sector_sv: np.ndarray,         # normalized sector statevector
    psi_exact: np.ndarray,         # exact GS in sector basis
    evecs: np.ndarray,             # eigenvectors from eigsh, shape (dim, k)
    evals: np.ndarray,             # eigenvalues, shape (k,)
    times: np.ndarray,
    phases: np.ndarray,
) -> tuple[float, float]:
    """
    Returns (fidelity_with_GS, postselection_probability).
    This is the honest analytic prediction of what the Hadamard-test
    circuit produces when all ancillas are postselected on |0>.
    """
    # Complex overlaps — no .real truncation
    coeffs = evecs.conj().T @ sector_sv          # shape (k,)

    # Cosine filter weights per eigenstate
    weights = np.ones(len(evals), dtype=float)
    for t, phi in zip(times, phases):
        weights *= np.cos(evals * t + phi)

    # Filtered (unnormalized) coefficients
    filtered = coeffs * weights                   # shape (k,)

    # Postselection probability = norm^2 of filtered state
    # (this is your Lambda^2 from the paper)
    prob = float(np.real(np.dot(filtered.conj(), filtered)))

    if prob < 1e-14:
        return 0.0, prob

    # Normalized filtered state in sector basis
    filtered_sv = evecs @ filtered
    filtered_sv /= np.sqrt(prob)

    fidelity = float(np.abs(np.dot(psi_exact.conj(), filtered_sv)) ** 2)
    return fidelity, prob

In [36]:
"""
main.py
=======
Single entry point comparing UCJ and DMRG ground-state circuits on the same
Heisenberg lattice.

Global config
-------------
  J1, J2    : Heisenberg couplings (defined below)
  LATTICE   : BaseLattice object shared by both methods

Design contract
---------------
  * Exact ground-state energy, eigenvector, and full spectrum come from a
    single Lanczos call (get_ground_state from ucj_circuit.py).  DMRG only
    adds its own variational energy; it does NOT re-run the diagonalisation.
  * Both circuits are transpiled to TARGET_BASIS before gate counts are printed.
  * Both circuits have the time-filter appended (append_filter from
    time_filter.py) using Lanczos energy gaps.
  * Metrics reported:
      - Fidelity with exact GS  before  filter
      - Fidelity with exact GS  after   filter
      - Max bond dimension used by DMRG
      - Gate counts (per gate type) for each circuit in TARGET_BASIS
"""

from __future__ import annotations

# ── stdlib ────────────────────────────────────────────────────────────────────
import textwrap
from typing import Optional

# ── numerics ──────────────────────────────────────────────────────────────────
import numpy as np
from scipy.sparse.linalg import eigsh

# ── Qiskit ────────────────────────────────────────────────────────────────────
from qiskit import transpile, QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp
'''
# ── project modules ───────────────────────────────────────────────────────────
from lattices import BaseLattice, make_lattice          # shared lattice layer

# UCJ: imports every public symbol so nothing is re-implemented here
from ucj_circuit import (
    _get_n_up,
    _build_basis,
    _build_hamiltonian_op,
    get_ground_state,                  # ← single Lanczos call lives here
    build_jax_hamiltonian,
    build_jastrow_fn,
    build_givens_pairs,
    build_ucj,
    TARGET_BASIS as UCJ_TARGET_BASIS,
    VARIANT      as UCJ_VARIANT,
    K_LAYERS     as UCJ_K_LAYERS,
)

# DMRG: imports every public symbol so nothing is re-implemented here
from dmrg_mps_circuit import (
    build_hamiltonian   as dmrg_build_hamiltonian,
    build_basis         as dmrg_build_basis,
    HeisenbergLattice,
    run_dmrg,
    mps_to_circuits,
    _transpile          as dmrg_transpile,
    BASIS_GATES         as DMRG_BASIS_GATES,
    DMRG_CHI_MAX,
)

# Time filter: imports every public symbol
from time_filter import (
    FilterBuilder,
    append_filter,
    new_func_v4,
    new_func_v5,
    fixtimes,
    unpack,
    timesconstraints,
    probability_constraints,
    probability_constraintsb,
)
'''
# =============================================================================
# ── GLOBAL CONFIGURATION ─────────────────────────────────────────────────────
# =============================================================================

J1       = 1.0
J2       = 0.7

# Define the lattice once; both methods share it.
# Change the arguments below to switch geometry / system size.
N_SITES  = 12
LATTICE: BaseLattice = make_lattice("chain", L=N_SITES)

# Target gate basis for transpilation and gate-count reporting
TARGET_BASIS = ["cx", "rz", "h", "s", "sdg"]

# DMRG bond dimension cap
CHI_MAX = DMRG_CHI_MAX

# UCJ hyper-parameters
VARIANT  = 'g'    # 're' | 'im' | 'g'
K_LAYERS = 1

# Time-filter settings
FILTER_TOTAL_TIME = 20.0
FILTER_A          = 4      # min number of pulses to sweep
FILTER_B          = 8      # max number of pulses to sweep
FILTER_METHOD     = "v5"   # 'v3' | 'v4' | 'v5'
FILTER_MAXITER    = 5000
FILTER_FTOL       = 1e-12
TROTTER_STEPS     = 1      # Trotter steps per filter pulse

# =============================================================================
# ── HELPERS ───────────────────────────────────────────────────────────────────
# =============================================================================

_DIVIDER = "─" * 72


def _header(title: str) -> None:
    print(f"\n{_DIVIDER}")
    print(f"  {title}")
    print(_DIVIDER)


def _print_gate_counts(label: str, qc: QuantumCircuit) -> None:
    ops   = qc.count_ops()
    total = sum(ops.values())
    print(f"\n  [{label}]  qubits={qc.num_qubits}  depth={qc.depth()}  "
          f"total_gates={total}")
    for gate in TARGET_BASIS:
        print(f"    {gate:<8} {ops.get(gate, 0)}")
    others = {g: c for g, c in ops.items() if g not in TARGET_BASIS}
    if others:
        print("    --- non-basis gates (not yet decomposed) ---")
        for gate, cnt in sorted(others.items(), key=lambda x: -x[1]):
            print(f"    {gate:<8} {cnt}")


def _statevector_from_circuit(qc: QuantumCircuit) -> np.ndarray:
    """Simulate the circuit and return the statevector as a numpy array."""
    sv = Statevector(qc)
    return np.array(sv)


def _fidelity_with_sector(
    sv_full: np.ndarray,
    psi_exact: np.ndarray,
    basis: np.ndarray,
    n: int,
) -> float:
    """
    Project the full 2^n statevector onto the fixed-Sz sector and compute
    |<psi_exact | psi_sector>|^2.

    psi_exact lives in the sector basis (length = len(basis)).
    sv_full   lives in the full 2^n Hilbert space.
    """
    sector_sv = sv_full[basis]                      # pick sector amplitudes
    norm = np.linalg.norm(sector_sv)
    if norm < 1e-14:
        return 0.0
    sector_sv = sector_sv / norm                    # normalise within sector
    overlap   = np.dot(np.conj(psi_exact), sector_sv)
    return float(np.abs(overlap) ** 2)


def _build_hamiltonian_pauli(lattice: BaseLattice,
                             j1: float, j2: float) -> SparsePauliOp:
    """
    Build a SparsePauliOp for the Heisenberg J1-J2 model so that
    append_filter can use Qiskit's PauliEvolutionGate.
    """
    n = lattice.n_sites
    pauli_list = []

    def _zz(i: int, j: int, coeff: float) -> None:
        label = ["I"] * n
        label[i] = "Z"
        label[j] = "Z"
        pauli_list.append(("".join(reversed(label)), coeff * 0.25))

    def _xx(i: int, j: int, coeff: float) -> None:
        label = ["I"] * n
        label[i] = "X"
        label[j] = "X"
        pauli_list.append(("".join(reversed(label)), coeff * 0.5))

    def _yy(i: int, j: int, coeff: float) -> None:
        label = ["I"] * n
        label[i] = "Y"
        label[j] = "Y"
        pauli_list.append(("".join(reversed(label)), coeff * 0.5))

    for (i, j) in lattice.nn_edges:
        _zz(i, j, j1); _xx(i, j, j1); _yy(i, j, j1)
    for (i, j) in lattice.nnn_edges:
        _zz(i, j, j2); _xx(i, j, j2); _yy(i, j, j2)

    return SparsePauliOp.from_list(pauli_list).simplify()


def _full_spectrum_gaps(
    lattice: BaseLattice,
    j1: float,
    j2: float,
    n_eigvals: int = 20,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Return (energies, gaps) where gaps = E_k - E_0,
    computed from the k lowest eigenstates of the sector Hamiltonian.
    This reuses _build_hamiltonian_op from ucj_circuit.py.
    """
    n    = lattice.n_sites
    n_up = _get_n_up(n)
    op, basis, _ = _build_hamiltonian_op(
        n, n_up, lattice.nn_edges, lattice.nnn_edges, j1, j2)
    k    = min(n_eigvals, len(basis) - 1)
    evals, _ = eigsh(op, k=k, which="SA", tol=1e-10, maxiter=20_000)
    evals    = np.sort(evals.real)
    return evals, evals - evals[0]


def _coeffs_sq_in_eigenbasis(
    psi_trial: np.ndarray,
    lattice: BaseLattice,
    j1: float,
    j2: float,
    n_eigvals: int = 20,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:  # ← added evecs
    n    = lattice.n_sites
    n_up = _get_n_up(n)
    op, basis, _ = _build_hamiltonian_op(
        n, n_up, lattice.nn_edges, lattice.nnn_edges, j1, j2)
    k = min(n_eigvals, len(basis) - 1)
    evals, evecs = eigsh(op, k=k, which="SA", tol=1e-10, maxiter=20_000)
    order  = np.argsort(evals.real)
    evals  = evals[order].real
    evecs  = evecs[:, order]
    coeffs = evecs.conj().T @ psi_trial   # ← was (evecs.T @ ...).real
    return evals, np.abs(coeffs) ** 2, coeffs, evecs


# =============================================================================
# ── LANCZOS (single call) ─────────────────────────────────────────────────────
# =============================================================================

def run_lanczos(
    lattice: BaseLattice,
    j1: float,
    j2: float,
) -> tuple[float, np.ndarray, np.ndarray, np.ndarray]:
    """
    Single Lanczos diagonalisation shared by UCJ and DMRG.

    Returns
    -------
    e0        : ground-state energy
    psi_exact : ground-state vector in sector basis
    basis     : integer sector basis (length = dim)
    idx_map   : dict mapping integer bitstring → row index
    """
    _header("Lanczos exact diagonalisation  (single call)")
    n    = lattice.n_sites
    n_up = _get_n_up(n)
    e0, psi_exact, basis, idx_map = get_ground_state(
        n, n_up,
        lattice.nn_edges, lattice.nnn_edges,
        j1, j2,
    )
    print(f"  N={n}  n_up={n_up}  sector_dim={len(basis)}")
    print(f"  E0       = {e0:.10f}")
    print(f"  E0/site  = {e0/n:.10f}")
    return e0, psi_exact, basis, idx_map


# =============================================================================
# ── UCJ RUN ───────────────────────────────────────────────────────────────────
# =============================================================================

def run_ucj_comparison(
    lattice: BaseLattice,
    j1: float,
    j2: float,
    e0: float,
    psi_exact: np.ndarray,
    basis: np.ndarray,
    hamiltonian_pauli: SparsePauliOp,
    energies: np.ndarray,
    gaps: np.ndarray,
) -> dict:

    _header("UCJ circuit construction + optimisation")
    n = lattice.n_sites

    tqc_ucj, gate_counts_ucj, depth_ucj = build_ucj(
        lattice, j1, j2, variant=VARIANT, k_layers=K_LAYERS,
        basis_gates=TARGET_BASIS,
    )

    _header("UCJ — pre-filter fidelity")
    sv_ucj      = _statevector_from_circuit(tqc_ucj)
    fid_pre_ucj = _fidelity_with_sector(sv_ucj, psi_exact, basis, n)
    print(f"  UCJ fidelity (pre-filter)  = {fid_pre_ucj:.8f}")

    _header("UCJ — time filter optimisation")
    sector_sv_ucj = sv_ucj[basis]
    norm          = np.linalg.norm(sector_sv_ucj)
    sector_sv_ucj = sector_sv_ucj / (norm + 1e-30)
    overlap_ucj   = float(np.abs(np.dot(np.conj(psi_exact), sector_sv_ucj)))

    # ── CHANGE 1: unpack the new 4th return value ──────────────────────────
    evals_filter, coeffs_sq_ucj, coeffs_ucj, evecs_filter = \
        _coeffs_sq_in_eigenbasis(sector_sv_ucj, lattice, j1, j2)

    fb_ucj = FilterBuilder(
        total_time = FILTER_TOTAL_TIME,
        energies   = evals_filter,
        overlap    = overlap_ucj,
        a          = FILTER_A,
        b          = FILTER_B,
        maxiter    = FILTER_MAXITER,
        ftol       = FILTER_FTOL,
        coeffs_sq  = coeffs_sq_ucj,
    )
    filter_results_ucj = fb_ucj.build(method=FILTER_METHOD)
    best_ucj = _best_filter(filter_results_ucj, sector_sv_ucj, psi_exact, evecs_filter, evals_filter)
    print(f"\n  Best filter:  ntimes={best_ucj['ntimes']}  "
          f"fun={best_ucj['fun']:.4e}  success={best_ucj['success']}")

    # ── Step 4 unchanged — keep the Hadamard-test circuit for gate counts ──
    qc_ucj_filtered = transpile(
        build_filter_circuit(
            tqc_ucj, hamiltonian_pauli,
            best_ucj["times"], best_ucj["phases"],
            trotter_steps=TROTTER_STEPS,
        ),
        basis_gates=TARGET_BASIS,
        optimization_level=0,
    )

    # ── CHANGE 2: analytic post-filter fidelity (honest Hadamard-test prediction)
    _header("UCJ — post-filter fidelity (analytic, Hadamard test)")
    fid_post_ucj, postsel_prob = post_filter_fidelity_analytic(
        sector_sv_ucj, psi_exact,
        evecs_filter, evals_filter,
        best_ucj["times"], best_ucj["phases"],
    )
    print(f"  UCJ fidelity (post-filter, analytic)  = {fid_post_ucj:.8f}")
    print(f"  Postselection probability              = {postsel_prob:.6f}")

    gate_counts_filtered_ucj = dict(qc_ucj_filtered.count_ops())

    return {
        "qc_raw":               tqc_ucj,
        "qc_filtered":          qc_ucj_filtered,
        "fidelity_pre":         fid_pre_ucj,
        "fidelity_post":        fid_post_ucj,
        "gate_counts_raw":      gate_counts_ucj,
        "gate_counts_filtered": gate_counts_filtered_ucj,
        "postsel_prob":         postsel_prob
    }

# =============================================================================
# ── DMRG RUN ──────────────────────────────────────────────────────────────────
# =============================================================================

def run_dmrg_comparison(
    lattice: BaseLattice,
    j1: float,
    j2: float,
    e0: float,
    psi_exact: np.ndarray,
    basis: np.ndarray,
    hamiltonian_pauli: SparsePauliOp,
    energies: np.ndarray,
    gaps: np.ndarray,
) -> dict:
    _header("DMRG ground state")
    n = lattice.n_sites

    # ── 1. DMRG ───────────────────────────────────────────────────────────────
    dmrg_E, psi_mps = run_dmrg(lattice, j1, j2, chi_max=CHI_MAX)
    max_bond_dim    = int(max(psi_mps.chi))
    print(f"  DMRG energy   = {dmrg_E:.10f}")
    print(f"  Exact energy  = {e0:.10f}")
    print(f"  |ΔE|          = {abs(dmrg_E - e0):.4e}")
    print(f"  Max bond dim  = {max_bond_dim}")

    # ── 2. MPS → circuits ─────────────────────────────────────────────────────
    _header("DMRG — MPS → circuits")
    exact_qc, approx_qc = mps_to_circuits(psi_mps, n)

    # ── 3. Pre-filter fidelities ──────────────────────────────────────────────
    _header("DMRG — pre-filter fidelities")
    sv_exact  = _statevector_from_circuit(exact_qc)
    sv_approx = _statevector_from_circuit(approx_qc)

    fid_exact_pre  = _fidelity_with_sector(sv_exact,  psi_exact, basis, n)
    fid_approx_pre = _fidelity_with_sector(sv_approx, psi_exact, basis, n)
    print(f"  DMRG exact  fidelity (pre-filter)  = {fid_exact_pre:.8f}")
    print(f"  DMRG approx fidelity (pre-filter)  = {fid_approx_pre:.8f}")

    # ── 4. Sector statevectors (normalised) ───────────────────────────────────
    sector_sv_exact = sv_exact[basis]
    sector_sv_exact = sector_sv_exact / (np.linalg.norm(sector_sv_exact) + 1e-30)

    sector_sv_approx = sv_approx[basis]
    sector_sv_approx = sector_sv_approx / (np.linalg.norm(sector_sv_approx) + 1e-30)

    # ── 5. Filter for exact circuit ───────────────────────────────────────────
    _header("DMRG — time filter optimisation  (exact MPS circuit)")
    evals_exact, coeffs_sq_exact, coeffs_exact, evecs_exact = \
        _coeffs_sq_in_eigenbasis(sector_sv_exact, lattice, j1, j2)

    fb_exact = FilterBuilder(
        total_time = FILTER_TOTAL_TIME,
        energies   = evals_exact,
        overlap    = float(np.abs(np.dot(np.conj(psi_exact), sector_sv_exact))),
        a          = FILTER_A,
        b          = FILTER_B,
        maxiter    = FILTER_MAXITER,
        ftol       = FILTER_FTOL,
        coeffs_sq  = coeffs_sq_exact,
    )
    filter_results_exact = fb_exact.build(method=FILTER_METHOD)
    best_exact = _best_filter(filter_results_exact, sector_sv_exact, psi_exact, evecs_exact, evals_exact)
    print(f"\n  Best filter:  ntimes={best_exact['ntimes']}  "
          f"fun={best_exact['fun']:.4e}  success={best_exact['success']}")

    # ── 6. Filter for approx circuit (its own optimisation) ───────────────────
    _header("DMRG — time filter optimisation  (approx MPS circuit)")
    evals_approx, coeffs_sq_approx, coeffs_approx, evecs_approx = \
        _coeffs_sq_in_eigenbasis(sector_sv_approx, lattice, j1, j2)

    fb_approx = FilterBuilder(
        total_time = FILTER_TOTAL_TIME,
        energies   = evals_approx,
        overlap    = float(np.abs(np.dot(np.conj(psi_exact), sector_sv_approx))),
        a          = FILTER_A,
        b          = FILTER_B,
        maxiter    = FILTER_MAXITER,
        ftol       = FILTER_FTOL,
        coeffs_sq  = coeffs_sq_approx,
    )
    filter_results_approx = fb_approx.build(method=FILTER_METHOD)
    best_approx = _best_filter(filter_results_approx, sector_sv_approx, psi_exact, evecs_approx, evals_approx)
    print(f"\n  Best filter:  ntimes={best_approx['ntimes']}  "
          f"fun={best_approx['fun']:.4e}  success={best_approx['success']}")

    # ── 7. Build Hadamard-test filter circuits ────────────────────────────────
    qc_exact_filtered  = transpile(build_filter_circuit(
        exact_qc, hamiltonian_pauli,
        best_exact["times"], best_exact["phases"],
        trotter_steps=TROTTER_STEPS),
        basis_gates=TARGET_BASIS,
        optimization_level=0,)

    qc_approx_filtered = transpile(build_filter_circuit(
        approx_qc, hamiltonian_pauli,
        best_approx["times"], best_approx["phases"],
        trotter_steps=TROTTER_STEPS),basis_gates=TARGET_BASIS,
        optimization_level=0,)

    # ── 8. Post-filter fidelities (analytic, Hadamard-test prediction) ────────
    _header("DMRG — post-filter fidelities (analytic, Hadamard test)")
    fid_exact_post, prob_exact = post_filter_fidelity_analytic(
        sector_sv_exact, psi_exact,
        evecs_exact, evals_exact,
        best_exact["times"], best_exact["phases"],
    )
    print(f"  DMRG exact  fidelity (post-filter) = {fid_exact_post:.8f}  "
          f"postsel_prob={prob_exact:.6f}")

    fid_approx_post, prob_approx = post_filter_fidelity_analytic(
        sector_sv_approx, psi_exact,
        evecs_approx, evals_approx,
        best_approx["times"], best_approx["phases"],
    )
    print(f"  DMRG approx fidelity (post-filter) = {fid_approx_post:.8f}  "
          f"postsel_prob={prob_approx:.6f}")

    return {
        "qc_exact_raw":               exact_qc,
        "qc_approx_raw":              approx_qc,
        "qc_exact_filtered":          qc_exact_filtered,
        "qc_approx_filtered":         qc_approx_filtered,
        "fidelity_exact_pre":         fid_exact_pre,
        "fidelity_approx_pre":        fid_approx_pre,
        "fidelity_exact_post":        fid_exact_post,
        "fidelity_approx_post":       fid_approx_post,
        "postsel_prob_exact":         prob_exact,
        "postsel_prob_approx":        prob_approx,
        "max_bond_dim":               max_bond_dim,
        "dmrg_energy":                dmrg_E,
        "gate_counts_exact_raw":      dict(exact_qc.count_ops()),
        "gate_counts_approx_raw":     dict(approx_qc.count_ops()),
        "gate_counts_exact_filtered": dict(qc_exact_filtered.count_ops()),
        "gate_counts_approx_filtered":dict(qc_approx_filtered.count_ops()),
    }


# =============================================================================
# ── SUMMARY PRINTER ───────────────────────────────────────────────────────────
# =============================================================================

def print_summary(
    e0: float,
    n: int,
    ucj_res: dict,
    dmrg_res: dict,
) -> None:
    _header("GATE COUNTS  —  target basis: " + str(TARGET_BASIS))

    _print_gate_counts("UCJ  raw (pre-filter)",      ucj_res["qc_raw"])
    _print_gate_counts("UCJ  filtered",              ucj_res["qc_filtered"])
    _print_gate_counts("DMRG exact  raw",            dmrg_res["qc_exact_raw"])
    _print_gate_counts("DMRG exact  filtered",       dmrg_res["qc_exact_filtered"])
    _print_gate_counts("DMRG approx raw",            dmrg_res["qc_approx_raw"])
    _print_gate_counts("DMRG approx filtered",       dmrg_res["qc_approx_filtered"])

    _header("METRICS SUMMARY")
    row_fmt = "  {:<38} {}"
    print(row_fmt.format("Exact E0 (Lanczos)",
                         f"{e0:.10f}"))
    print(row_fmt.format("Exact E0/site",
                         f"{e0/n:.10f}"))
    print()
    print(row_fmt.format("DMRG energy",
                         f"{dmrg_res['dmrg_energy']:.10f}"))
    print(row_fmt.format("DMRG |ΔE| vs Lanczos",
                         f"{abs(dmrg_res['dmrg_energy'] - e0):.4e}"))
    print(row_fmt.format("DMRG max bond dimension",
                         dmrg_res["max_bond_dim"]))
    print()
    print(row_fmt.format("UCJ  fidelity  pre-filter",
                         f"{ucj_res['fidelity_pre']:.8f}"))
    print(row_fmt.format("UCJ  fidelity  post-filter",
                         f"{ucj_res['fidelity_post']:.8f}"))
    print()
    print(row_fmt.format("DMRG exact  fidelity  pre-filter",
                         f"{dmrg_res['fidelity_exact_pre']:.8f}"))
    print(row_fmt.format("DMRG exact  fidelity  post-filter",
                         f"{dmrg_res['fidelity_exact_post']:.8f}"))
    print()
    print(row_fmt.format("DMRG approx fidelity  pre-filter",
                         f"{dmrg_res['fidelity_approx_pre']:.8f}"))
    print(row_fmt.format("DMRG approx fidelity  post-filter",
                         f"{dmrg_res['fidelity_approx_post']:.8f}"))

    print(row_fmt.format("UCJ  postselection probability",
                     f"{ucj_res['postsel_prob']:.6f}"))
    print(row_fmt.format("UCJ  shots per postselection",
                        f"{int(np.ceil(1/ucj_res['postsel_prob']))}"))
    print()
    print(row_fmt.format("DMRG exact  postsel. probability",
                        f"{dmrg_res['postsel_prob_exact']:.6f}"))
    print(row_fmt.format("DMRG approx postsel. probability",
                        f"{dmrg_res['postsel_prob_approx']:.6f}"))
    print(row_fmt.format("DMRG approx shots per postselection",
                        f"{int(np.ceil(1/dmrg_res['postsel_prob_approx']))}"))
    print()
    print(_DIVIDER)


# =============================================================================
# ── ENTRY POINT ───────────────────────────────────────────────────────────────
# =============================================================================

def main() -> tuple[dict, dict]:
    """
    Run the full UCJ-vs-DMRG comparison pipeline.

    Execution order
    ---------------
    1.  Lanczos         — exact GS energy + statevector (single call)
    2.  Lanczos gaps    — low-lying spectrum for filter optimisation
    3.  UCJ             — variational optimisation → circuit
    4.  DMRG            — TeNPy ground state → MPS circuits
    5.  Time filter     — optimise + append to all circuits
    6.  Gate counts     — print per gate type in TARGET_BASIS
    7.  Metrics         — fidelity (pre/post filter), max bond dim

    Returns
    -------
    ucj_res  : dict with UCJ results (see run_ucj_comparison)
    dmrg_res : dict with DMRG results (see run_dmrg_comparison)
    """
    _header(f"UCJ vs DMRG  —  N={N_SITES}  J1={J1}  J2={J2}  "
            f"lattice={LATTICE.name}")

    n = LATTICE.n_sites

    # ── 1. Single Lanczos call ────────────────────────────────────────────────
    e0, psi_exact, basis, idx_map = run_lanczos(LATTICE, J1, J2)

    # ── 2. Low-lying spectrum (energy gaps for filter) ────────────────────────
    _header("Low-lying spectrum for time-filter energy gaps")
    energies, gaps = _full_spectrum_gaps(LATTICE, J1, J2, n_eigvals=20)
    print(f"  Lowest {len(energies)} eigenvalues:")
    for k, (e, g) in enumerate(zip(energies, gaps)):
        print(f"    k={k:2d}  E={e:.8f}  gap={g:.8f}")

    # ── 3. Pauli Hamiltonian (needed by append_filter) ────────────────────────
    hamiltonian_pauli = _build_hamiltonian_pauli(LATTICE, J1, J2)

    # ── 4. UCJ ────────────────────────────────────────────────────────────────
    ucj_res = run_ucj_comparison(
        LATTICE, J1, J2,
        e0, psi_exact, basis,
        hamiltonian_pauli, energies, gaps,
    )

    # ── 5. DMRG ───────────────────────────────────────────────────────────────
    dmrg_res = run_dmrg_comparison(
        LATTICE, J1, J2,
        e0, psi_exact, basis,
        hamiltonian_pauli, energies, gaps,
    )

    # ── 6. Summary ────────────────────────────────────────────────────────────
    print_summary(e0, n, ucj_res, dmrg_res)

    return ucj_res, dmrg_res


if __name__ == "__main__":
    ucj_res, dmrg_res = main()

[chain L=12]  nn=12 (2/site)  nnn=12 (2/site)  ✓

────────────────────────────────────────────────────────────────────────
  UCJ vs DMRG  —  N=12  J1=1.0  J2=0.7  lattice=chain L=12
────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────
  Lanczos exact diagonalisation  (single call)
────────────────────────────────────────────────────────────────────────
  N=12  n_up=6  sector_dim=924
  E0       = -4.8085762851
  E0/site  = -0.4007146904

────────────────────────────────────────────────────────────────────────
  Low-lying spectrum for time-filter energy gaps
────────────────────────────────────────────────────────────────────────
  Lowest 20 eigenvalues:
    k= 0  E=-4.80857629  gap=0.00000000
    k= 1  E=-4.70907101  gap=0.09950527
    k= 2  E=-4.22111060  gap=0.58746568
    k= 3  E=-4.21145869  gap=0.59711760
    k= 4  E=-4.21145869  gap=0.59711760
    k= 5  E=-4.11507671  gap=0.69349958
    k

/usr/local/lib/python3.12/dist-packages/tenpy/tools/params.py:243: UserWarning: unused options for config HeisenbergLattice:
['bc_MPS', 'conserve']
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
/usr/local/lib/python3.12/dist-packages/tenpy/networks/mps.py:1629: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites, bc, unit_cell_width)


[DMRG]  lattice=chain L=12  E=-4.8085762851  E/site=-0.4007146904  chi_max=64
  DMRG energy   = -4.8085762851
  Exact energy  = -4.8085762851
  |ΔE|          = 8.8818e-16
  Max bond dim  = 64

────────────────────────────────────────────────────────────────────────
  DMRG — MPS → circuits
────────────────────────────────────────────────────────────────────────

[circuit:exact]  qubits=12  depth=64647  gates=115002
  sdg   41020
  rz    40379
  h     21640
  cx    11963

[circuit:approximate]  qubits=12  depth=1130  gates=8245
  sdg   3520
  rz    2640
  h     1760
  cx    325

────────────────────────────────────────────────────────────────────────
  DMRG — pre-filter fidelities
────────────────────────────────────────────────────────────────────────
  DMRG exact  fidelity (pre-filter)  = 1.00000000
  DMRG approx fidelity (pre-filter)  = 0.56943345

────────────────────────────────────────────────────────────────────────
  DMRG — time filter optimisation  (exact MPS circuit)
──────────